# Phase 5 v3 — Fine-Tuning (5 Critical Fixes + Speed Optimizations)

**v2 → v3 changes (evidence-backed — each fix has web references):**

1. **Response-only loss masking** — v2 trained on entire conversation (Google default behavior — confirmed by Google employee [tiffanychen on AI Forum](https://discuss.ai.google.dev/t/medgemma-finetuning-padding-and-labels-masking/105422)). QLoRA paper Table 10 shows completions-only training improves MMLU by 2-5% across 4 datasets. v3 masks everything before `<start_of_turn>model\n`.
2. **REMOVED `modules_to_save`** — PEFT GitHub issues [#1750](https://github.com/huggingface/peft/issues/1750) and [#2864](https://github.com/huggingface/peft/issues/2864) confirm: passing both `lm_head` AND `embed_tokens` to `modules_to_save` **UNTIES** them (creates separate tensor copies). PEFT maintainer BenjaminBossan recommends: "not add the embeddings to modules_to_save." Contributor romitjain confirms: "Target embedding layer only. The LM head is simply pointing to this layer."
3. **3 epochs** (was 1) — v2 had only 94 steps total. Google's official notebook uses 9000 samples × 1 epoch = 562 steps. 3 epochs + early stopping (patience=3).
4. **LR 1e-4** (was 2e-4) — half rate to preserve pretrained histopathology features. 2e-4 at 3 epochs caused catastrophic forgetting (MSI AUC DROPPED from 0.66 → 0.50).
5. **MSI-H oversampling 5×** — only ~2% of training samples are MSI-H. v2 model predicted ALL MSS (AUC=0.50). Oversampling prevents majority-class collapse.

**Speed optimizations (target: ~20 min, was ~40 min):**
- `max_length`: 2048 → 512 (our sequences are ~350 tokens; [HF Forum confirms](https://discuss.huggingface.co/t/sfttrainer-training-very-slow-is-this-training-speed-expected/115458) this is the **biggest** speed win)
- `batch_size`: 4 → 8 (A100 80GB handles this easily with shorter seq)
- `grad_accum`: 4 → 2 (effective batch stays 16)
- `eval_batch_size`: 2 → 8 (faster evals)
- Eval frequency: 3×/epoch → 1×/epoch

**What stayed the same (Google-aligned):**
- LoRA r=16, alpha=16, all-linear, QLoRA 4-bit NF4
- SFTTrainer, adamw_torch_fused, gradient_checkpointing
- A100 optimizations: Flash Attention 2, TF32, image pre-caching, local SSD
- Padding: right for training, left for inference

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
TRAINING_DIR = f"{DATA_DIR}/training_v2"  # Reuse v2 training data (format is fine)
MODEL_DIR = f"{PROJECT_DIR}/models/immunopath-v3"
RESULTS_DIR = f"{PROJECT_DIR}/results/phase5_v3"

# Local SSD paths — critical for speed
LOCAL_CKPT_DIR = "/content/immunopath_ckpts_v3"
LOCAL_LOG_DIR = "/content/immunopath_logs_v3"

for d in [MODEL_DIR, RESULTS_DIR, LOCAL_CKPT_DIR, LOCAL_LOG_DIR]:
    os.makedirs(d, exist_ok=True)

import subprocess
subprocess.run([
    "pip", "install", "-q", "--no-cache-dir",
    "transformers>=4.50.0",
    "accelerate>=0.34.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.44.0",
    "trl>=0.12.0",
    "datasets",
    "tqdm",
], check=True)

try:
    subprocess.run(["pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=True)
    FLASH_ATTN_AVAILABLE = True
    print("Flash Attention 2 installed ✓")
except Exception:
    FLASH_ATTN_AVAILABLE = False
    print("Flash Attention not available — using eager attention")

from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get("HF_TOKEN"))
    print("HuggingFace login ✓")
except Exception:
    print("Set HF_TOKEN in Colab Secrets")

import torch
import gc

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    if "A100" in gpu_name:
        print("  ✓ A100 detected — TF32 + Flash Attention enabled")
else:
    print("No GPU detected!")

import transformers, peft, trl
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")

## Configuration (v3 changes + speed optimizations highlighted)

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    model_id: str = "google/medgemma-1.5-4b-it"

    # LoRA — same as Google's official notebook
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    target_modules: str = "all-linear"

    # V3 FIX #2: REMOVED modules_to_save
    # PEFT issues #1750 and #2864 confirm: passing both lm_head AND embed_tokens
    # to modules_to_save UNTIES them (creates separate tensor copies).
    # PEFT maintainer BenjaminBossan: "not add the embeddings to modules_to_save"
    # romitjain: "Target embedding layer only. The LM head is simply pointing to this layer."
    # All prediction tokens ("high", "low", "MSS", etc.) are already in vocabulary.
    modules_to_save: list = field(default_factory=list)  # EMPTY — was ["lm_head", "embed_tokens"]

    # V3 FIX #3: 3 epochs (was 1)
    # v2 had only 94 steps — not enough for value learning.
    # Google used 562 steps on 9000 samples. 3 epochs × 1502 samples = 282 steps.
    num_epochs: int = 3
    # Speed optimization: larger batch + lower accum (A100 80GB handles it at 512 max_length)
    batch_size: int = 8       # was 4 — 2× more samples per step
    grad_accum: int = 2       # was 4 — effective batch stays 16 (8×2)

    # V3 FIX #4: LR 1e-4 (was 2e-4)
    # Half rate to preserve pretrained histopathology features over 3 epochs.
    lr: float = 1e-4

    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 0.3
    lr_scheduler_type: str = "linear"

    # V3 FIX #5: MSI-H oversampling factor
    msi_h_oversample: int = 5

    # Data — max_length reduced for speed (HF Forum confirms this is #1 speed optimization)
    # Our sequences are ~350 tokens (prompt ~150 + response ~200). 512 gives plenty of headroom.
    max_patches: int = 4
    max_length: int = 512     # was 2048 — ~4× less padding computation

    # Paths
    train_path: str = f"{TRAINING_DIR}/train.jsonl"
    val_path: str = f"{TRAINING_DIR}/val.jsonl"
    output_dir: str = MODEL_DIR
    log_dir: str = RESULTS_DIR

    logging_steps: int = 10

cfg = Config()
print(f"Config (v3):")
print(f"  LoRA: r={cfg.lora_r}, alpha={cfg.lora_alpha}, target={cfg.target_modules}")
print(f"  modules_to_save: {cfg.modules_to_save} ← EMPTY (PEFT #1750/#2864 fix)")
print(f"  Batch: {cfg.batch_size}×{cfg.grad_accum} = {cfg.batch_size*cfg.grad_accum} effective (optimized for speed)")
print(f"  LR: {cfg.lr} ← was 2e-4")
print(f"  Epochs: {cfg.num_epochs} ← was 1")
print(f"  MSI-H oversample: {cfg.msi_h_oversample}×")
print(f"  max_length: {cfg.max_length} ← was 2048 (biggest speed win)")
print(f"  max_grad_norm: {cfg.max_grad_norm}")

## Pre-Copy Data to Local SSD

In [ ]:
import json, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

LOCAL_TRAIN_DIR = "/content/immunopath_local_v3"
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

print("Pre-copying training data to local SSD...")

# Fallback to training/ if training_v2/ doesn't exist
if not os.path.exists(cfg.train_path):
    print(f"{cfg.train_path} not found, trying training/ directory...")
    TRAINING_DIR_FALLBACK = f"{DATA_DIR}/training"
    cfg.train_path = f"{TRAINING_DIR_FALLBACK}/train.jsonl"
    cfg.val_path = f"{TRAINING_DIR_FALLBACK}/val.jsonl"

# Copy JSONL files
for split in ["train.jsonl", "val.jsonl"]:
    src = cfg.train_path.replace("train.jsonl", split)
    dst = f"{LOCAL_TRAIN_DIR}/{split}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied {split}")

# Collect all patch files
DRIVE_PREFIX = "/content/drive/MyDrive/ImmunoPath"
patch_files = set()
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                if p.startswith(DRIVE_PREFIX):
                    patch_files.add(p)

print(f"  Total patch files referenced: {len(patch_files)}")

# Create directories
dst_dirs = set()
for src_file in patch_files:
    dst_dirs.add(os.path.dirname(src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)))
for d in dst_dirs:
    os.makedirs(d, exist_ok=True)

# Copy only missing files
to_copy = []
for src_file in patch_files:
    dst_file = src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)
    if not os.path.exists(dst_file):
        to_copy.append((src_file, dst_file))

print(f"  Need to copy: {len(to_copy)} (already cached: {len(patch_files) - len(to_copy)})")

def _copy_one(pair):
    shutil.copy2(pair[0], pair[1])

if to_copy:
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(tqdm(ex.map(_copy_one, to_copy), total=len(to_copy), desc="Copying patches"))

# Rewrite paths in JSONL to point to local SSD
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    updated = []
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            sample["patch_paths"] = [
                p.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR) if p.startswith(DRIVE_PREFIX) else p
                for p in sample["patch_paths"]
            ]
            updated.append(json.dumps(sample))
    with open(local_path, "w") as f:
        f.write("\n".join(updated) + "\n")

# Validate
missing = []
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample["patch_paths"][:cfg.max_patches]:
                if not os.path.exists(p):
                    missing.append(p)
if missing:
    print(f"  ⚠ {len(missing)} patches missing")
else:
    print("  All patches verified ✓")

cfg.train_path = f"{LOCAL_TRAIN_DIR}/train.jsonl"
cfg.val_path = f"{LOCAL_TRAIN_DIR}/val.jsonl"
print(f"\nSSD pre-copy complete. Training from {LOCAL_TRAIN_DIR}")

## Pre-Cache ALL Images to RAM

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import time

IMAGE_CACHE = {}

def _load_image(path):
    try:
        img = Image.open(path).convert("RGB")
        img.load()
        return path, img
    except Exception:
        return path, None

all_patch_paths = set()
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                all_patch_paths.add(p)

print(f"Pre-caching {len(all_patch_paths)} unique patches to RAM...")
cache_start = time.time()

with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(tqdm(
        pool.map(_load_image, all_patch_paths),
        total=len(all_patch_paths),
        desc="Caching images"
    ))

for path, img in results:
    if img is not None:
        IMAGE_CACHE[path] = img

cache_elapsed = time.time() - cache_start
cache_mb = sum(img.size[0] * img.size[1] * 3 for img in IMAGE_CACHE.values()) / 1e6
print(f"\n  Cached {len(IMAGE_CACHE)}/{len(all_patch_paths)} images in {cache_elapsed:.1f}s")
print(f"  RAM usage: ~{cache_mb:.0f} MB")

## Load Model

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f"Loading {cfg.model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

attn_impl = "flash_attention_2" if FLASH_ATTN_AVAILABLE else "eager"

model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
    quantization_config=bnb_config,
)

processor = AutoProcessor.from_pretrained(cfg.model_id)
processor.tokenizer.padding_side = "right"  # Right padding for training

allocated = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded ({attn_impl}), VRAM: {allocated:.2f} GB")
print(f"Tokenizer padding_side: {processor.tokenizer.padding_side}")
print(f"tie_word_embeddings: {model.config.tie_word_embeddings}")

## LoRA Config (v3: NO modules_to_save — PEFT #1750/#2864)

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    r=cfg.lora_r,
    bias="none",
    target_modules=cfg.target_modules,
    task_type="CAUSAL_LM",
    # V3 FIX #2: NO modules_to_save
    # PEFT #1750: "Passing both embed_tokens and lm_head to modules_to_save UNTIES them"
    # PEFT #2864: With default ensure_weight_tying=False, both in modules_to_save = "treat as separate"
    # PEFT maintainer: "not add the embeddings to modules_to_save"
    # This corrupted v2's predictions. LoRA alone is sufficient for our task.
)

print(f"LoRA Config (v3):")
print(f"  r={peft_config.r}, alpha={peft_config.lora_alpha}")
print(f"  target_modules: {peft_config.target_modules}")
print(f"  modules_to_save: {peft_config.modules_to_save}  ← REMOVED (v3 fix)")
print(f"  All learning happens through LoRA adapters on internal layers")

## Build Dataset (with MSI-H Oversampling)

In [ ]:
import json
from datasets import Dataset
from typing import Any


def load_jsonl(path: str) -> list:
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line.strip()))
    return records


def oversample_minority_classes(records: list, msi_factor: int = 5) -> list:
    """V3 FIX #5: Oversample MSI-H samples to prevent majority-class collapse.

    v2 had ~2% MSI-H → model predicted ALL MSS (AUC=0.50, zero discrimination).
    5x oversampling brings MSI-H from ~30 to ~150 samples in 1502-sample dataset.
    """
    msi_h_samples = []
    msi_h_count = 0
    mss_count = 0
    tme_counts = {}

    for r in records:
        try:
            resp = json.loads(r["response"])
            msi = resp.get("msi_status", "").upper()
            tme = resp.get("tme_subtype", "")
            if msi == "MSI-H":
                msi_h_samples.append(r)
                msi_h_count += 1
            elif msi == "MSS":
                mss_count += 1
            if tme:
                tme_counts[tme] = tme_counts.get(tme, 0) + 1
        except Exception:
            pass

    print(f"  Original class distribution:")
    print(f"    MSI-H: {msi_h_count}, MSS: {mss_count}")
    print(f"    TME: {tme_counts}")

    # Oversample MSI-H
    extra = []
    for _ in range(msi_factor - 1):
        extra.extend(msi_h_samples)

    # Also oversample underrepresented TME subtypes
    if tme_counts:
        max_tme = max(tme_counts.values())
        for tme_label, count in tme_counts.items():
            if count < max_tme // 3:  # If less than 1/3 of majority
                tme_samples = [
                    r for r in records
                    if json.loads(r["response"]).get("tme_subtype") == tme_label
                ]
                # Oversample to reach 1/3 of majority
                n_extra = (max_tme // 3) - count
                repeats = max(1, n_extra // max(1, len(tme_samples)))
                for _ in range(repeats):
                    extra.extend(tme_samples)
                print(f"    Oversampled TME={tme_label}: +{repeats * len(tme_samples)} samples")

    result = records + extra
    print(f"  After oversampling: {len(result)} samples (+{len(extra)} added)")

    # Verify new distribution
    new_msi_h = sum(1 for r in result if json.loads(r["response"]).get("msi_status", "").upper() == "MSI-H")
    print(f"  New MSI-H count: {new_msi_h} ({100*new_msi_h/len(result):.1f}%)")

    return result


def format_for_sft(example: dict[str, Any]) -> dict[str, Any]:
    n_images = min(len(example.get("patch_paths", [])), cfg.max_patches)
    if n_images == 0:
        n_images = 1

    user_content = [{"type": "image"} for _ in range(n_images)]
    user_content.append({"type": "text", "text": example["prompt"]})

    example["messages"] = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": example["response"]}]},
    ]
    return example


# Load raw records
print("Loading datasets...")
train_records = load_jsonl(cfg.train_path)
val_records = load_jsonl(cfg.val_path)
print(f"  Raw train: {len(train_records)}, val: {len(val_records)}")

# V3 FIX #5: Oversample minority classes in training set
print("\nApplying MSI-H oversampling...")
train_records = oversample_minority_classes(train_records, cfg.msi_h_oversample)

# Convert to HuggingFace Dataset
train_data = Dataset.from_list(train_records)
val_data = Dataset.from_list(val_records)

print("\nFormatting for SFT...")
train_data = train_data.map(format_for_sft, desc="Formatting train")
val_data = val_data.map(format_for_sft, desc="Formatting val")

# Shuffle training data (mixes oversampled records)
train_data = train_data.shuffle(seed=42)

print(f"\nFinal datasets:")
print(f"  Train: {len(train_data)} samples")
print(f"  Val:   {len(val_data)} samples")

## Custom Collator (v3: Response-Only Loss Masking)

**THE critical v3 fix — evidence-backed.**

**Background:** [Google AI Forum](https://discuss.ai.google.dev/t/medgemma-finetuning-padding-and-labels-masking/105422) — Google employee tiffanychen confirmed: *"The notebook follows the default behavior of the SFTTrainer where the model is trained on both the prompt and answers... You could also decide to train on completions only."*

**Evidence:** [QLoRA paper Table 10](https://arxiv.org/abs/2305.14314) — training on completions only improves MMLU by 2-5% across ALL 4 tested datasets (Unnatural Instructions, Alpaca, FLAN v2, Chip2).

**Alternative:** TRL v0.22.0+ supports `assistant_only_loss=True` natively for VLMs. We use a custom collator instead because it also handles image caching and VLM-specific processing.

| | v2 (Google default) | v3 (completions-only) |
|---|---|---|
| Prompt tokens in loss | ✅ Yes (~150) | ❌ Masked |
| Response tokens in loss | ✅ Yes (~200) | ✅ Yes (~200) |
| Learning signal focus | Diluted across template | 100% on predictions |

In [ ]:
from typing import Any

# Pre-compute the response template token sequence (done ONCE)
# In Gemma 3 chat format: <start_of_turn>model\n is the response preamble
_RESPONSE_TEMPLATE_IDS = processor.tokenizer.encode(
    "<start_of_turn>model\n", add_special_tokens=False
)
print(f"Response template token IDs: {_RESPONSE_TEMPLATE_IDS}")
print(f"  Decoded back: '{processor.tokenizer.decode(_RESPONSE_TEMPLATE_IDS)}'")

# Pre-compute special token IDs for masking
_BOI_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(
    processor.tokenizer.special_tokens_map.get("boi_token", "<start_of_image>")
)
_PAD_TOKEN_ID = processor.tokenizer.pad_token_id
_IMAGE_PLACEHOLDER_ID = 262144

print(f"  pad_token_id: {_PAD_TOKEN_ID}")
print(f"  boi_token_id: {_BOI_TOKEN_ID}")
print(f"  image_placeholder_id: {_IMAGE_PLACEHOLDER_ID}")


def _find_response_start(input_ids_list: list, template_ids: list) -> int:
    """Find the LAST occurrence of the response template in token sequence.

    Returns the index AFTER the template (i.e., where the actual response starts).
    Returns -1 if not found.
    """
    tlen = len(template_ids)
    pos = -1
    for i in range(len(input_ids_list) - tlen + 1):
        if input_ids_list[i:i + tlen] == template_ids:
            pos = i + tlen  # Position AFTER the template
    return pos


def collate_fn(examples: list[dict[str, Any]]):
    """v3 Collator: Response-only loss masking + cached images.

    1. Build text via apply_chat_template (tokenize=False)
    2. Joint tokenization with processor(text=..., images=...)
    3. Create labels = input_ids.clone()
    4. Mask pad + image tokens (same as v2/Google)
    5. V3 FIX: ADDITIONALLY mask everything BEFORE the response
    """
    texts = []
    all_images = []

    for example in examples:
        # Load images from cache (zero I/O)
        images = []
        for p in example.get("patch_paths", [])[:cfg.max_patches]:
            if p in IMAGE_CACHE:
                images.append(IMAGE_CACHE[p])
            else:
                try:
                    images.append(Image.open(p).convert("RGB"))
                except Exception:
                    pass
        if not images:
            images = [Image.new("RGB", (512, 512), "white")]

        all_images.append(images)

        text = processor.apply_chat_template(
            example["messages"],
            add_generation_prompt=False,
            tokenize=False,
        ).strip()
        texts.append(text)

    # Joint tokenization
    batch = processor(
        text=texts,
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_length,
    )

    labels = batch["input_ids"].clone()

    # Step 1: Mask pad + image tokens (same as v2/Google)
    if _PAD_TOKEN_ID is not None:
        labels[labels == _PAD_TOKEN_ID] = -100
    if _BOI_TOKEN_ID is not None:
        labels[labels == _BOI_TOKEN_ID] = -100
    labels[labels == _IMAGE_PLACEHOLDER_ID] = -100

    # Step 2: V3 FIX #1 — Mask everything BEFORE the response
    for i in range(labels.shape[0]):
        ids_list = batch["input_ids"][i].tolist()
        response_start = _find_response_start(ids_list, _RESPONSE_TEMPLATE_IDS)
        if response_start > 0:
            labels[i, :response_start] = -100
        # else: template not found — fall back to v2 behavior (train on everything)

    batch["labels"] = labels
    return batch


# === VERIFY the masking works on one sample ===
print("\n--- Verifying response-only masking ---")
sample = train_data[0]
test_batch = collate_fn([sample])
ids = test_batch["input_ids"][0]
lbls = test_batch["labels"][0]

total_tokens = ids.shape[0]
masked_tokens = (lbls == -100).sum().item()
trained_tokens = total_tokens - masked_tokens

print(f"  Total tokens:   {total_tokens}")
print(f"  Masked (-100):  {masked_tokens} ({100*masked_tokens/total_tokens:.0f}%)")
print(f"  Trained on:     {trained_tokens} ({100*trained_tokens/total_tokens:.0f}%)")

# Show what the model trains on
trained_ids = ids[lbls != -100]
trained_text = processor.tokenizer.decode(trained_ids, skip_special_tokens=True)
print(f"  Response text:  {trained_text[:300]}")

# v2 comparison
v2_trained = total_tokens - (ids == _PAD_TOKEN_ID).sum().item() - (ids == _IMAGE_PLACEHOLDER_ID).sum().item()
print(f"\n  v2 would train on: {v2_trained} tokens")
print(f"  v3 trains on:      {trained_tokens} tokens")
print(f"  Signal improvement: {v2_trained/max(1,trained_tokens):.1f}× more focused")

## Training Configuration + SFTTrainer (Speed-Optimized)

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
import time

# Clean previous checkpoints
if os.path.exists(LOCAL_CKPT_DIR):
    existing = [d for d in os.listdir(LOCAL_CKPT_DIR) if d.startswith("checkpoint-")]
    if existing:
        print(f"Cleaning {len(existing)} old checkpoints...")
        for d in existing:
            shutil.rmtree(os.path.join(LOCAL_CKPT_DIR, d), ignore_errors=True)

torch.cuda.empty_cache()
gc.collect()

# Estimate steps for eval scheduling
steps_per_epoch = len(train_data) // (cfg.batch_size * cfg.grad_accum)
total_steps = steps_per_epoch * cfg.num_epochs
# Speed: eval once per epoch (was 3× per epoch = 9 total evals)
eval_every = max(steps_per_epoch, 15)
print(f"Steps per epoch: ~{steps_per_epoch}")
print(f"Total steps: ~{total_steps}")
print(f"Eval every {eval_every} steps (~{total_steps // eval_every} evaluations total)")
print(f"  [Speed opt: 1×/epoch vs 3×/epoch in v3-original]")

training_args = SFTConfig(
    output_dir=LOCAL_CKPT_DIR,

    # V3 FIX #3: 3 epochs (was 1)
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=8,   # was 2 — speed optimization
    gradient_accumulation_steps=cfg.grad_accum,

    # V3 FIX #4: LR 1e-4 (was 2e-4)
    learning_rate=cfg.lr,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.max_grad_norm,
    lr_scheduler_type=cfg.lr_scheduler_type,
    optim="adamw_torch_fused",
    bf16=True,

    # Eval/Save — more frequent than v2
    logging_steps=cfg.logging_steps,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=3,

    # Logging
    report_to="tensorboard",
    logging_dir=f"{LOCAL_LOG_DIR}/tb_logs",

    # Best model + early stopping
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Dataset handling
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],

    # Data loading
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=4,
)

# Early stopping: stop if eval_loss doesn't improve for 3 consecutive evals
early_stopping = EarlyStoppingCallback(early_stopping_patience=3)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
    callbacks=[early_stopping],
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"\nSFTTrainer ready:")
print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"  Effective batch: {cfg.batch_size * cfg.grad_accum}")
print(f"  Train samples: {len(train_data)} (with oversampling)")
print(f"  Val samples: {len(val_data)}")
print(f"  Epochs: {cfg.num_epochs}, steps/epoch: ~{steps_per_epoch}")
print(f"  LR: {cfg.lr}, scheduler: {cfg.lr_scheduler_type}")
print(f"  Early stopping patience: 3 evaluations")

print(f"\n  V3 FIXES ACTIVE (evidence-backed):")
print(f"  ✓ Response-only loss masking (QLoRA paper: +2-5% MMLU)")
print(f"  ✓ No modules_to_save (PEFT #1750/#2864: avoids tied weight untying)")
print(f"  ✓ 3 epochs + early stopping")
print(f"  ✓ LR 1e-4 (preserves pretrained features)")
print(f"  ✓ MSI-H oversampling {cfg.msi_h_oversample}×")
print(f"  SPEED OPTS: max_length={cfg.max_length}, batch={cfg.batch_size}×{cfg.grad_accum}, eval_batch=8")

## TRAIN

In [ ]:
print("=" * 60)
print("STARTING FINE-TUNING (v3 — 5 Critical Fixes + Speed Optimized)")
print("=" * 60)
print(f"  Model:     {cfg.model_id}")
print(f"  LoRA:      r={cfg.lora_r}, alpha={cfg.lora_alpha}")
print(f"  target:    {cfg.target_modules}")
print(f"  modules_to_save: NONE (PEFT #1750/#2864 fix)")
print(f"  Batch:     {cfg.batch_size}×{cfg.grad_accum}={cfg.batch_size*cfg.grad_accum}")
print(f"  max_length: {cfg.max_length} (from 2048 — biggest speed win)")
print(f"  Epochs:    {cfg.num_epochs} + early stopping")
print(f"  LR:        {cfg.lr}")
print(f"  Loss mask: Response-only (QLoRA paper +2-5% MMLU)")
print("=" * 60)

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE in {elapsed/60:.1f} minutes (target: ~20 min)")
print(f"Final train loss: {train_result.training_loss:.4f}")
print(f"{'=' * 60}")

## Save LoRA Adapters

In [ ]:
# Save locally first (fast), then copy to GDrive
local_adapter_dir = f"{LOCAL_CKPT_DIR}/lora_adapters"
os.makedirs(local_adapter_dir, exist_ok=True)

trainer.save_model(local_adapter_dir)
processor.save_pretrained(local_adapter_dir)
print(f"Adapters saved locally: {local_adapter_dir}")

# Copy to GDrive
print("\nCopying adapters to GDrive...")
gdrive_adapter_dir = f"{cfg.output_dir}/lora_adapters"
os.makedirs(gdrive_adapter_dir, exist_ok=True)

for f_name in os.listdir(local_adapter_dir):
    src = os.path.join(local_adapter_dir, f_name)
    dst = os.path.join(gdrive_adapter_dir, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"  → {gdrive_adapter_dir}")

# Also copy to model root
for f_name in os.listdir(local_adapter_dir):
    src = os.path.join(local_adapter_dir, f_name)
    dst = os.path.join(cfg.output_dir, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"  → {cfg.output_dir}")

saved_files = os.listdir(gdrive_adapter_dir)
print(f"\nAdapter files: {saved_files}")
print("✓ Adapters saved to both local SSD and GDrive")

## Save Training Log

In [ ]:
from datetime import datetime

log_history = trainer.state.log_history
train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs = [l for l in log_history if "eval_loss" in l]

training_report = {
    "phase": "5_v3",
    "timestamp": datetime.now().isoformat(),
    "model_id": cfg.model_id,
    "v3_fixes": [
        "Response-only loss masking — QLoRA paper Table 10: +2-5% MMLU. Google Forum confirms default was train-on-all.",
        "REMOVED modules_to_save — PEFT #1750 + #2864: both lm_head+embed_tokens UNTIES them. Maintainer recommends removal.",
        "3 epochs + EarlyStoppingCallback(patience=3)",
        "LR 1e-4 (was 2e-4 — preserves pretrained features over 3 epochs)",
        f"MSI-H oversampling {cfg.msi_h_oversample}× (prevents class collapse)",
    ],
    "speed_optimizations": [
        "max_length 2048→512 (sequences ~350 tokens; HF Forum confirms #1 speed win)",
        "batch_size 4→8 (A100 80GB with shorter sequences)",
        "grad_accum 4→2 (effective batch stays 16)",
        "eval_batch_size 2→8",
        "eval frequency 3×/epoch → 1×/epoch",
    ],
    "evidence_sources": [
        "https://discuss.ai.google.dev/t/medgemma-finetuning-padding-and-labels-masking/105422",
        "https://github.com/huggingface/peft/issues/1750",
        "https://github.com/huggingface/peft/issues/2864",
        "https://arxiv.org/abs/2305.14314 (QLoRA paper, Table 10)",
        "https://discuss.huggingface.co/t/sfttrainer-training-very-slow-is-this-training-speed-expected/115458",
        "https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora",
    ],
    "peft_config": {
        "r": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": cfg.target_modules,
        "modules_to_save": "NONE (PEFT #1750/#2864 fix)",
    },
    "training_config": {
        "batch_size": cfg.batch_size,
        "grad_accum": cfg.grad_accum,
        "effective_batch": cfg.batch_size * cfg.grad_accum,
        "lr": cfg.lr,
        "epochs": cfg.num_epochs,
        "warmup_ratio": cfg.warmup_ratio,
        "max_grad_norm": cfg.max_grad_norm,
        "lr_scheduler": cfg.lr_scheduler_type,
        "optimizer": "adamw_torch_fused",
        "loss_masking": "response-only (prompt + image + pad masked)",
        "early_stopping_patience": 3,
    },
    "data": {
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "msi_h_oversample_factor": cfg.msi_h_oversample,
        "image_cache_size": len(IMAGE_CACHE),
    },
    "results": {
        "final_train_loss": train_result.training_loss,
        "best_eval_loss": min((l["eval_loss"] for l in eval_logs), default=None),
        "training_time_minutes": round(elapsed / 60, 1),
        "total_steps": trainer.state.global_step,
        "stopped_early": trainer.state.global_step < total_steps,
    },
    "train_logs": train_logs,
    "eval_logs": eval_logs,
}

local_log_path = f"{LOCAL_LOG_DIR}/training_log.json"
with open(local_log_path, 'w') as f:
    json.dump(training_report, f, indent=2, default=str)

gdrive_log_path = f"{RESULTS_DIR}/training_log.json"
shutil.copy2(local_log_path, gdrive_log_path)
print(f"Training log saved to {gdrive_log_path}")

best_eval = training_report["results"]["best_eval_loss"]
print(f"\n  Final train loss:  {train_result.training_loss:.4f}")
print(f"  Best eval loss:    {best_eval}")
print(f"  Training time:     {elapsed/60:.1f} min")
print(f"  Total steps:       {trainer.state.global_step}")
print(f"  Stopped early:     {training_report['results']['stopped_early']}")

## Quick Inference Test

In [ ]:
# Switch to left padding for inference
processor.tokenizer.padding_side = "left"

val_jsonl = f"{LOCAL_TRAIN_DIR}/val.jsonl"
val_record = None
if os.path.exists(val_jsonl):
    with open(val_jsonl) as f:
        val_record = json.loads(f.readline().strip())

if val_record:
    images = []
    for p in val_record["patch_paths"][:cfg.max_patches]:
        if p in IMAGE_CACHE:
            images.append(IMAGE_CACHE[p])
        else:
            try:
                images.append(Image.open(p).convert("RGB"))
            except Exception:
                pass
    if not images:
        images = [Image.new("RGB", (512, 512), "white")]

    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": val_record["prompt"]})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(p.device for p in trainer.model.parameters() if p.device.type != 'meta')
    inputs = {
        k: v.to(device, dtype=torch.bfloat16) if v.is_floating_point()
        else v.to(device)
        for k, v in inputs.items()
        if torch.is_tensor(v)
    }
    input_len = inputs["input_ids"].shape[-1]

    trainer.model.eval()
    with torch.inference_mode():
        output_ids = trainer.model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True,
        )

    generated = output_ids[0][input_len:]
    response_text = processor.decode(generated, skip_special_tokens=True).strip()

    print("INFERENCE TEST (v3):")
    print(f"  Output: {response_text[:400]}")
    print()

    try:
        clean = response_text
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start != -1 and end > start:
            parsed = json.loads(clean[start:end])
            print(f"  Valid JSON: {list(parsed.keys())}")
            print(f"  Parsed: {json.dumps(parsed, indent=2)}")
        else:
            print("  No JSON found in output")
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")

    print(f"\n  Ground truth:")
    gt = json.loads(val_record["response"])
    print(f"  {json.dumps(gt, indent=2)}")
else:
    print("No val samples available")

processor.tokenizer.padding_side = "right"

## Phase 5 v3 Summary

In [ ]:
print("\n" + "=" * 60)
print("PHASE 5 v3 — FINE-TUNING COMPLETE")
print("=" * 60)
print(f"\n  Model:       {cfg.model_id}")
print(f"  PEFT:        LoRA (r={cfg.lora_r}, α={cfg.lora_alpha})")
print(f"  Train:       {len(train_data)} samples × {cfg.num_epochs} epochs")
print(f"  Val:         {len(val_data)} samples")
print(f"  Final loss:  {train_result.training_loss:.4f}")
print(f"  Best eval:   {best_eval}")
print(f"  Time:        {elapsed/60:.1f} min (target: ~20 min)")
print(f"  Steps:       {trainer.state.global_step}")
print(f"  Adapters:    {gdrive_adapter_dir}")

print(f"\n  V3 FIXES APPLIED (all evidence-backed):")
print(f"  1. Response-only loss masking (QLoRA paper: +2-5% MMLU) ✓")
print(f"  2. No modules_to_save (PEFT #1750/#2864: untied weights fixed) ✓")
print(f"  3. 3 epochs + early stopping ✓")
print(f"  4. LR 1e-4 (was 2e-4) ✓")
print(f"  5. MSI-H oversampling {cfg.msi_h_oversample}× ✓")
print(f"\n  SPEED OPTIMIZATIONS:")
print(f"  • max_length: 2048 → {cfg.max_length} (biggest win)")
print(f"  • batch: {cfg.batch_size}×{cfg.grad_accum} (was 4×4)")
print(f"  • eval_batch: 8 (was 2)")
print(f"  • eval freq: 1×/epoch (was 3×/epoch)")

print(f"\n{'=' * 60}")
print("NEXT: Phase 6 v3 — Evaluation")
print(f"  Load from: {gdrive_adapter_dir}")
print(f"{'=' * 60}")